In [ ]:
import requests

url = "https://cache-datasets.s3.amazonaws.com/cache_dataset_txt/2008_msr/msr-cambridge2.tar"
filename = 'msrdataset.tar'

with requests.get(url, stream=True) as r:
    r.raise_for_status()
    with open(filename, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

In [ ]:
import tarfile
import io
import gzip

target_filename = 'MSR-Cambridge/src1_0.csv.gz'

with tarfile.open(filename, 'r') as tar:
    # Get the file-like object for the specific member
    member = tar.getmember(target_filename)
    f = tar.extractfile(member)

    if f is not None:
        # Since the file inside the tar is gzipped, decompress it first
        with gzip.open(f, 'rt', encoding='utf-8') as ifh, open('partialdataset.csv', 'w', encoding='utf-8') as ofh:
            # ifh is already a text-mode file object that reads decompressed content
            text_stream = ifh

            header = text_stream.readline()
            ofh.write(header)

            for i, line in enumerate(text_stream):
                if i < 100000:
                     ofh.write(line)
                else:
                    break

In [ ]:
import tarfile
import io
import gzip

target_filename = 'MSR-Cambridge/src1_1.csv.gz'

with tarfile.open(filename, 'r') as tar:
    # Get the file-like object for the specific member
    member = tar.getmember(target_filename)
    f = tar.extractfile(member)

    if f is not None:
        # Since the file inside the tar is gzipped, decompress it first
        with gzip.open(f, 'rt', encoding='utf-8') as ifh, open('testdataset.csv', 'w', encoding='utf-8') as ofh:
            # ifh is already a text-mode file object that reads decompressed content
            text_stream = ifh

            header = text_stream.readline()
            ofh.write(header)

            for i, line in enumerate(text_stream):
                if i < 50000:
                     ofh.write(line)
                else:
                    break

In [ ]:
import pandas as pd


cache_partition = 64 # Define cache partition here

df = pd.read_csv('partialdataset.csv', header=None).iloc[1:100001].reset_index(drop=True)

df = df.rename(columns={df.columns[4]: 'offset', df.columns[5]: 'size'})

df['partition_id'] = df.index // cache_partition

unique_offsets = df.drop_duplicates(subset=['partition_id', 'offset'])

cav = unique_offsets.groupby('partition_id')['size'].sum().tolist()

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('testdataset.csv', header=None).iloc[1:20001].reset_index(drop=True)

df = df.rename(columns={df.columns[4]: 'offset', df.columns[5]: 'size'})

df['partition_id'] = df.index // cache_partition

unique_offsets = df.drop_duplicates(subset=['partition_id', 'offset'])

cav_test = unique_offsets.groupby('partition_id')['size'].sum().tolist()


In [ ]:
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np

scalar = MinMaxScaler(feature_range=(0.1, 0.9))

timestampvals = pd.read_csv('partialdataset.csv', usecols=[0], nrows=100000).values
timestamps = scalar.fit_transform(timestampvals)

offsetvals = pd.read_csv('partialdataset.csv', usecols=[4], nrows=100000).values
offset = scalar.fit_transform(offsetvals)

sizevals = pd.read_csv('partialdataset.csv', usecols=[5], nrows=100000).values
size = scalar.fit_transform(sizevals)

responsetimevals = pd.read_csv('partialdataset.csv', usecols=[6], nrows=100000).values
responsetime = scalar.fit_transform(responsetimevals)

r_wdf = pd.read_csv('partialdataset.csv', usecols=[3], nrows=100000)
r_w = r_wdf.iloc[:, 0].map({'Write': 0, 'Read': 1}).values

cav_scaled = scalar.fit_transform(np.array(cav).reshape(-1, 1))

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np

scalar = MinMaxScaler(feature_range=(0.1, 0.9))

timestampvals = pd.read_csv('testdataset.csv', usecols=[0], nrows=20000).values
timestamps_test = scalar.fit_transform(timestampvals)

offsetvals = pd.read_csv('testdataset.csv', usecols=[4], nrows=20000).values
offset_test = scalar.fit_transform(offsetvals)

sizevals = pd.read_csv('testdataset.csv', usecols=[5], nrows=20000).values
size_test = scalar.fit_transform(sizevals)

responsetimevals = pd.read_csv('testdataset.csv', usecols=[6], nrows=20000).values
responsetime_test = scalar.fit_transform(responsetimevals)

r_wdf = pd.read_csv('testdataset.csv', usecols=[3], nrows=20000)
r_w_test = r_wdf.iloc[:, 0].map({'Write': 0, 'Read': 1}).values

cav_test_scaled = scalar.fit_transform(np.array(cav_test).reshape(-1, 1))

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, LSTM, Dense, TimeDistributed, Dropout

time_steps = cache_partition

features = np.stack([timestamps, r_w.reshape(-1, 1), offset, size, responsetime], axis=1)
#features = np.stack([offset, size], axis=1)

features_test = np.stack([timestamps_test, r_w_test.reshape(-1, 1), offset_test, size_test, responsetime_test], axis=1)
#features_test = np.stack([offset_test, size_test], axis=1)


channels = features.shape[1]
height = 1
width = 1

num_total_samples = features.shape[0]
num_test_samples = features_test.shape[0]
num_batches = num_total_samples // time_steps
num_batches_test = num_test_samples // time_steps

# Trim features to be divisible by time_steps
features_trimmed = features[:num_batches * time_steps]
features_test_trimmed = features_test[:num_batches_test * time_steps]

# Reshape X to (num_batches, time_steps, height, width, channels)
X = features_trimmed.reshape(num_batches, time_steps, height, width, channels)
X_test = features_test_trimmed.reshape(num_batches_test, time_steps, height, width, channels)

# Trim y to match X's batch count
y = np.array(cav_scaled[:num_batches])
y_test = np.array(cav_test_scaled[:num_batches_test])

# 2. CNN-LSTM Model Definition
model = Sequential()

# CNN Model Layers
model.add(TimeDistributed(Conv2D(64, (1, 1), activation='leaky_relu'), input_shape=(time_steps, height, width, channels)))
model.add(TimeDistributed(MaxPooling2D((1, 1))))
model.add(TimeDistributed(Conv2D(128, (1, 1), activation='leaky_relu')))
model.add(TimeDistributed(MaxPooling2D((1, 1))))
model.add(TimeDistributed(Flatten()))

# Sequence Learning: LSTM layers for temporal dynamics
model.add(LSTM(64, return_sequences=True))
model.add(LSTM(64, return_sequences=False))
model.add(Dropout(0.3))

# Output layer
model.add(Dense(128, activation='leaky_relu'))
model.add(Dense(1, activation='linear'))

# 3. Compile the Model
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
#model.summary()

# 4. Train the Model
model.fit(X, y, epochs=25, batch_size=4)
print("Model created successfully.")

# 5. Evaluate the model
loss, mae = model.evaluate(X_test, y_test, verbose=1)
predictions = model.predict(X_test)
MAPE = np.mean(np.abs(predictions - y_test) / (y_test))
accuracy = (1 - MAPE) * 100

print(f"Test Loss: {loss}, Test MAE: {mae}, Accuracy: {accuracy}")

model.summary()
np.set_printoptions(threshold=np.inf)
#print(predictions)
#print(y_test)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/wrapper.py:27: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/25
391/391 ━━━━━━━━━━━━━━━━━━━━ 30s 35ms/step - loss: 0.0175 - mae: 0.0930
Epoch 2/25
391/391 ━━━━━━━━━━━━━━━━━━━━ 13s 34ms/step - loss: 0.0049 - mae: 0.0536
Epoch 3/25
391/391 ━━━━━━━━━━━━━━━━━━━━ 13s 34ms/step - loss: 0.0039 - mae: 0.0474
Epoch 4/25
391/391 ━━━━━━━━━━━━━━━━━━━━ 14s 34ms/step - loss: 0.0029 - mae: 0.0418
Epoch 5/25
391/391 ━━━━━━━━━━━━━━━━━━━━ 13s 34ms/step - loss: 0.0023 - mae: 0.0374
Epoch 6/25
391/391 ━━━━━━━━━━━━━━━━━━━━ 20s 33ms/step - loss: 0.0025 - mae: 0.0388
Epoch 7/25
391/391 ━━━━━━━━━━━━━━━━━━━━ 13s 34ms/step - loss: 0.0027 - mae: 0.0404
Epoch 8/25
391/391 ━━━━━━━━━━━━━━━━━━━━ 21s 35ms/step - loss: 0.0027 - mae: 0.0398
Epoch 9/25
391/391 ━━━━━━━━━━━━━━━━━━━━ 13s 34ms/step - loss: 0.0018 - mae: 0.0331
Epoch 10/25
391/391 ━━━━━━━━━━━━━━━━━━━━ 13s 34ms/step - loss: 0.0014 - mae: 0.0290
Epoch 11/25
391/391 ━━━━━━━━━━━━━━━━━━━━ 13s 34ms/step - loss: 0.0013 - mae: 0.0280
Epoch 12/25
391/391 ━━━━━━━━━━━━━━━━━━━━ 13s 33ms/step - loss: 0.0011 - mae: 0.0255
E

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed                │ (None, 64, 1, 1, 64)   │           384 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 64, 1, 1, 64)   │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ (None, 64, 1, 1, 128)  │         8,320 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_3              │ (None, 64, 1, 1, 128)  │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_4              │ (None, 64, 128)        │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64, 64)         │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 298,757 (1.14 MB)

 Trainable params: 99,585 (389.00 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 199,172 (778.02 KB)

In [ ]:
import os

model_path = "./stock_model.keras"

model.save(model_path)

print(f"Stock model saved to: {model_path}")

# Correctly get the size of the saved model file
model_size_bytes = os.path.getsize(model_path)
model_size_mb = model_size_bytes / (1024 * 1024)

print(f"Size of the saved stock model: {model_size_mb:.2f} MB")

Stock model saved to: ./stock_model.keras
Size of the saved stock model: 1.20 MB


Iterative Pruning

In [ ]:
class ManualPruningCallback(tf.keras.callbacks.Callback):
    def __init__(self, masks):
        super().__init__()
        self.masks = masks

    def on_batch_end(self, batch, logs=None):
        for layer_name, mask in self.masks.items():
            layer = self.model.get_layer(layer_name)
            weights = layer.get_weights()
            weights[0] = weights[0] * mask
            layer.set_weights(weights)

In [ ]:
import numpy as np
import tensorflow as tf

def generate_pruning_masks(model, pruning_percentage):
    masks = {}
    for layer in model.layers:
        if (isinstance(layer, (tf.keras.layers.Conv2D, tf.keras.layers.Dense, tf.keras.layers.LSTM)) or
            (isinstance(layer, tf.keras.layers.TimeDistributed) and
             isinstance(layer.layer, (tf.keras.layers.Conv2D, tf.keras.layers.Dense, tf.keras.layers.LSTM)))):

            if len(layer.get_weights()) > 0: # Get kernel weights. Time Distributed layers' weights are accessed through the wrapper
                kernel_weights = layer.get_weights()[0]
            else:
                continue

            flat_weights = np.abs(kernel_weights).flatten() # Flatten to be used to find the threshold

            threshold = np.percentile(flat_weights, pruning_percentage * 100) # Calculate threshold based on pruning percentage

            mask = tf.cast(tf.math.greater(tf.math.abs(kernel_weights), threshold), tf.float32) # Generate mask: 1 for over the threshold, 0 for under the threshold

            masks[layer.name] = mask # Store mask with the name  of the layer as a key

    return masks


# You can adjust the pruning_percentage as needed (e.g., 0.1 for 10% pruning)
pruning_masks = generate_pruning_masks(model, pruning_percentage=0.9)

# Instantiate the ManualPruningCallback with the generated masks
manual_pruning_callback = ManualPruningCallback(pruning_masks)

# Recompile the model before fine-tuning with the callback
# It's important to recompile if you intend to continue training with the callback
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

print("Starting fine-tuning with ManualPruningCallback...")

# Fine-tune the model with the manual pruning callback
# You can adjust the number of epochs for fine-tuning
model.fit(X, y, epochs=5, batch_size=4, callbacks=[manual_pruning_callback], verbose=1)

print("Fine-tuning with ManualPruningCallback complete.")

# Evaluate the model after manual pruning
loss_manual_pruned, mae_manual_pruned = model.evaluate(X_test, y_test, verbose=1)
predictions_manual_pruned = model.predict(X_test)
MAPE_manual_pruned = np.mean(np.abs(predictions_manual_pruned - y_test) / (y_test)) # Added epsilon to prevent division by zero
accuracy_manual_pruned = (1 - MAPE_manual_pruned) * 100

print(f"\nManual Pruned Test Loss: {loss_manual_pruned:.4f}, Manual Pruned Test MAE: {mae_manual_pruned:.4f}, Manual Pruned Accuracy: {accuracy_manual_pruned:.2f}%")

model.summary()

Starting fine-tuning with ManualPruningCallback...
Epoch 1/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 35s 49ms/step - loss: 2.2789e-04 - mae: 0.0117
Epoch 2/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 20s 52ms/step - loss: 2.4081e-04 - mae: 0.0117
Epoch 3/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 19s 48ms/step - loss: 1.9926e-04 - mae: 0.0107
Epoch 4/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 20s 51ms/step - loss: 2.1194e-04 - mae: 0.0111
Epoch 5/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 19s 50ms/step - loss: 2.3128e-04 - mae: 0.0117
Fine-tuning with ManualPruningCallback complete.
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - loss: 8.4089e-04 - mae: 0.0244
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 324ms/step

Manual Pruned Test Loss: 0.0008, Manual Pruned Test MAE: 0.0244, Manual Pruned Accuracy: 94.98%


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed                │ (None, 64, 1, 1, 64)   │           384 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 64, 1, 1, 64)   │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ (None, 64, 1, 1, 128)  │         8,320 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_3              │ (None, 64, 1, 1, 128)  │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_4              │ (None, 64, 128)        │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64, 64)         │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 298,757 (1.14 MB)

 Trainable params: 99,585 (389.00 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 199,172 (778.02 KB)

In [ ]:
import os

pruned_model_path = "./pruned_model.keras"

# Save the model after pruning and fine-tuning
model.save(pruned_model_path)

print(f"Pruned model saved to: {pruned_model_path}")

# Get and display the size of the saved pruned model file
model_size_bytes_pruned = os.path.getsize(pruned_model_path)
model_size_mb_pruned = model_size_bytes_pruned / (1024 * 1024)

print(f"Size of the saved pruned model: {model_size_mb_pruned:.2f} MB")

Pruned model saved to: ./pruned_model.keras
Size of the saved pruned model: 1.20 MB


In [ ]:
import gzip
import shutil
import os

def compress_and_check_size(model_path):
    # 1. Define the output path
    compressed_path = model_path + ".gz"

    # 2. Compress the model file
    with open(model_path, 'rb') as f_in:
        with gzip.open(compressed_path, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)

    # 3. Get and display sizes
    original_size = os.path.getsize(model_path)
    compressed_size = os.path.getsize(compressed_path)

    print(f"Original Model Size: {original_size / 1024:.2f} KB")
    print(f"Compressed Model Size: {compressed_size / 1024:.2f} KB")
    print(f"Size Reduction: {100 * (1 - compressed_size / original_size):.2f}%")

# Example usage with the actual saved model
compress_and_check_size('pruned_model.keras')

Original Model Size: 1233.43 KB
Compressed Model Size: 857.27 KB
Size Reduction: 30.50%
